# 13 · HMDA：第二轮特征 × 最佳参数，五折确认并导出提交（Colab GPU）

12 的五折结论（快速配置，相对 68 列的 G0）：

|特征集|相对 G0|更好折数|
|---|---:|---:|
|**G7**：去掉地区频次 + 全部第二轮块（72 列）|**+0.00125**|5/5|
|LTV 申报精度（单独）|+0.00075|4/5|
|LTV 产品门槛（单独）|+0.00074|5/5|
|反推房价 / 期限（单独）|+0.0001 / +0.0002|3/5|

本 notebook 用 11 的协议（内层 15% 早停 → 外层重训；五个折模型测试概率平均作为提交）比较：

|组|参数|特征|
|---|---|---|
|`M0_C11_all`|C11|11 的 68 列（= 已提交的 K2，同轮对照）|
|`M1_C11_G7`|C11|G7（72 列）|
|`M2_C7_G7`|C7|G7|
|`M3_C11_lean`|C11|去掉频次 + 精度 + 门槛（68 列）|

三个新组五折都完成后，自动计算 **M1+M2+M3 等权融合**的五折 AUC，并写出 `submission_blend_M1_M2_M3.csv`。**选五折平均最高的提交**。

**操作**：Colab 选 **T4 GPU**，从上到下运行。每折约 7–9 分钟，全部约 2.5–3 小时；结果存 Drive `artifacts/final_gpu/<EXPERIMENT_ID>/`，断线后从头运行可续跑。

## 1. 环境、GPU 与配置

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PROJECT_DIR = Path(os.environ.get(
    'HMDA_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis',
)).expanduser().resolve()
PREPARED_DIR = PROJECT_DIR / 'artifacts' / 'prepared'
SEED = 42
CATBOOST_VERSION = '1.2.10'
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'找不到项目目录：{PROJECT_DIR}；请修改 PROJECT_DIR')

try:
    installed_catboost = importlib.metadata.version('catboost')
except importlib.metadata.PackageNotFoundError:
    installed_catboost = None
if installed_catboost != CATBOOST_VERSION:
    if 'catboost' in sys.modules:
        raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启 Colab 运行时后从头运行')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        f'catboost=={CATBOOST_VERSION}',
    ])
    importlib.invalidate_caches()

import catboost
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

if catboost.__version__ != CATBOOST_VERSION:
    raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('未找到 nvidia-smi：请在 Colab 更改运行时类型，选择 T4 GPU 后从头运行')
gpu_probe = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    text=True, capture_output=True, check=False,
)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError('NVIDIA GPU 不可用：请在 Colab 选择 T4 GPU 后从头运行')
GPU_NAME = gpu_probe.stdout.strip().splitlines()[0]

EXPERIMENT_ID = 'round2_final_v1'
FOLDS = [1, 2, 3, 4, 5]
RUN_ONLY = None
INNER_STOP_FRACTION = 0.15
MAX_ITERATIONS = 10000
OD_WAIT = 300
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'final_gpu' / EXPERIMENT_ID

GPU_BASE = dict(
    loss_function='Logloss', nan_mode='Min', random_seed=SEED,
    task_type='GPU', devices='0', thread_count=4,
    allow_writing_files=False, verbose=False,
)
ALL_BLOCKS = ['dti', 'ltv', 'income', 'missing', 'freq']
NO_FREQ = ['dti', 'ltv', 'income', 'missing']
FAST = dict(depth=8, learning_rate=0.05, l2_leaf_reg=3,
            bootstrap_type='Bernoulli', subsample=0.8, early_stop=False, iterations=1600)
C7 = dict(depth=8, learning_rate=0.03, l2_leaf_reg=10, bootstrap_type='Bernoulli', subsample=0.8)
C11 = dict(depth=6, learning_rate=0.03, l2_leaf_reg=10, bootstrap_type='Bernoulli', subsample=0.8)
ROUND2 = ['precision', 'thresholds', 'implied', 'term']
G7 = NO_FREQ + ROUND2                      # 12 中最好的特征集（72 列）
LEAN = NO_FREQ + ['precision', 'thresholds']  # 只保留 12 中单独有效的两块（68 列）
GROUPS = {
    'M0_C11_all': dict(params=C11, blocks=ALL_BLOCKS),   # = 11 的 K2（已提交 0.92073），同轮对照
    'M1_C11_G7': dict(params=C11, blocks=G7),
    'M2_C7_G7': dict(params=C7, blocks=G7),
    'M3_C11_lean': dict(params=C11, blocks=LEAN),
}
BASELINE = 'M0_C11_all'
BLEND_GROUPS = ['M1_C11_G7', 'M2_C7_G7', 'M3_C11_lean']   # 等权融合，五折都完成后自动计算
if RUN_ONLY is not None:
    unknown = set(RUN_ONLY) - set(GROUPS)
    if unknown:
        raise ValueError(f'RUN_ONLY 中有未知组：{sorted(unknown)}')
print(f'项目：{PROJECT_DIR}')
print(f'GPU：{GPU_NAME}；CatBoost {catboost.__version__}；折：{FOLDS}；输出：{OUTPUT_DIR}')

Mounted at /content/drive
项目：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis
GPU：Tesla T4, 15360 MiB；CatBoost 1.2.10；折：[1, 2, 3, 4, 5]；输出：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis/artifacts/final_gpu/round2_final_v1


## 2. 回读并核对 01 的数据契约（与 04 相同）

In [2]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


manifest_path = PREPARED_DIR / 'manifest.json'
schema_path = PREPARED_DIR / 'schema.json'
if not manifest_path.is_file() or not schema_path.is_file():
    raise FileNotFoundError(
        f'缺少 {manifest_path} 或 {schema_path}；请先运行 01_hmda_data_preparation.ipynb'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
schema = json.loads(schema_path.read_text(encoding='utf-8'))
require(manifest['schema_version'] == schema['schema_version'] == '1.0', '数据契约版本不匹配')
require(manifest['seed'] == SEED and manifest['n_splits'] == 5, '五折配置不匹配')
require(set(manifest['source_files']) == {'train.csv', 'test.csv', 'sample_submission.csv'},
        '原始文件清单不完整')
expected_prepared = {
    'train_features.csv', 'test_features.csv', 'train_labels.csv',
    'test_ids.csv', 'folds.csv', 'audit.csv', 'schema.json',
}
require(set(manifest['output_files']) == expected_prepared, '准备文件清单不完整')
for name, info in manifest['source_files'].items():
    path = PROJECT_DIR / name
    require(path.is_file(), f'缺少原始数据：{path}')
    require(sha256_file(path) == info['sha256'], f'原始文件已变化：{name}；请先重跑 01')
for name, info in manifest['output_files'].items():
    path = PREPARED_DIR / name
    require(path.is_file(), f'缺少准备文件：{path}')
    require(sha256_file(path) == info['sha256'], f'准备文件哈希不匹配：{name}；请先重跑 01')

FEATURE_COLUMNS = schema['feature_columns']
NUMERIC_COLUMNS = schema['numeric_columns']
CATEGORICAL_COLUMNS = schema['categorical_columns']
require(len(FEATURE_COLUMNS) == 56 and len(NUMERIC_COLUMNS) == 13
        and len(CATEGORICAL_COLUMNS) == 43, '预期为 56 列：13 数值、43 类别')
require(set(FEATURE_COLUMNS) == set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS),
        '数值/类别列分组不完整')
require(not {'id', 'accepted'} & set(FEATURE_COLUMNS), 'ID 或标签不能作为特征')


def read_features(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    require(frame.columns.tolist() == FEATURE_COLUMNS, f'特征列或顺序不匹配：{path.name}')
    for column in NUMERIC_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column].replace('', np.nan), errors='raise',
        ).astype('float64')
    require(not np.isinf(frame[NUMERIC_COLUMNS].to_numpy()).any(), '数值列含无穷大')
    require(not frame[CATEGORICAL_COLUMNS].eq('').any().any(), '类别列含空字符串')
    return frame


X_base = read_features(PREPARED_DIR / 'train_features.csv')
X_test_base = read_features(PREPARED_DIR / 'test_features.csv')
labels = pd.read_csv(PREPARED_DIR / 'train_labels.csv', dtype=str, keep_default_na=False)
test_id_frame = pd.read_csv(PREPARED_DIR / 'test_ids.csv', dtype=str, keep_default_na=False)
fold_frame = pd.read_csv(PREPARED_DIR / 'folds.csv', dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(PROJECT_DIR / 'sample_submission.csv', dtype=str, keep_default_na=False)

require(labels.columns.tolist() == ['id', 'accepted'], '训练标签列不匹配')
require(test_id_frame.columns.tolist() == ['id'], '测试 ID 列不匹配')
require(fold_frame.columns.tolist() == ['id', 'fold'], '折号文件列不匹配')
require(sample_submission.columns.tolist() == ['id', 'accepted'], '提交模板列不匹配')
require(len(X_base) == len(labels) == len(fold_frame) == 105000, '训练行数不匹配')
require(len(X_test_base) == len(test_id_frame) == len(sample_submission) == 45000,
        '测试行数不匹配')

train_ids = labels['id']
test_ids = test_id_frame['id']
require(train_ids.is_unique and test_ids.is_unique, 'ID 重复')
require(train_ids.ne('').all() and test_ids.ne('').all(), 'ID 为空')
require(train_ids.equals(fold_frame['id']), '折号 ID 与训练行错位')
require(test_ids.equals(sample_submission['id']), '测试 ID 与提交模板顺序不符')
require(set(train_ids).isdisjoint(test_ids), '训练与测试 ID 重叠')
require(set(labels['accepted']) == {'0', '1'}, 'accepted 需同时包含 0 和 1')
raw_train_identity = pd.read_csv(
    PROJECT_DIR / 'train.csv', usecols=['id', 'accepted'],
    dtype=str, keep_default_na=False,
)
raw_test_identity = pd.read_csv(
    PROJECT_DIR / 'test.csv', usecols=['id'], dtype=str, keep_default_na=False,
)
require(train_ids.equals(raw_train_identity['id'])
        and labels['accepted'].equals(raw_train_identity['accepted']),
        '准备标签与原始 train.csv 的 ID/标签不一致')
require(test_ids.equals(raw_test_identity['id']),
        '准备测试 ID 与原始 test.csv 不一致')
y = labels['accepted'].astype('int8')
require(set(fold_frame['fold']) == {'1', '2', '3', '4', '5'}, '折号应为 1..5')
fold_assignment = fold_frame['fold'].astype('int8').to_numpy()

expected_folds = np.zeros(len(X_base), dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for fold, (_, valid_index) in enumerate(splitter.split(X_base, y), start=1):
    expected_folds[valid_index] = fold
require(np.array_equal(fold_assignment, expected_folds), '折号并非固定 seed=42 五折')
DATA_HASHES = {
    'manifest.json': sha256_file(manifest_path),
    **{name: info['sha256'] for name, info in manifest['output_files'].items()},
}
print(f'数据核对通过：训练 {len(X_base):,}，测试 {len(X_test_base):,}，56 个原始特征。')
display(pd.crosstab(fold_assignment, y, rownames=['fold'], colnames=['accepted']))

def checked_probabilities(model, features):
    require(list(model.classes_) == [0, 1], '类别顺序不是 [0, 1]')
    probability = np.asarray(model.predict_proba(features)[:, 1], dtype=float)
    require(probability.ndim == 1 and len(probability) == len(features)
            and np.isfinite(probability).all()
            and ((probability >= 0) & (probability <= 1)).all(),
            '预测概率无效')
    return probability


def write_json(path, payload):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def fixed_config(output_dir, config):
    """首次运行写入配置；之后配置必须完全相同，避免把不同实验的结果混在一个目录。"""
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / 'config.json'
    if path.is_file():
        saved = json.loads(path.read_text(encoding='utf-8'))
        require(saved == config,
                f'{output_dir.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        write_json(path, config)

数据核对通过：训练 105,000，测试 45,000，56 个原始特征。


accepted,0,1
fold,,
1,3151,17849
2,3151,17849
3,3151,17849
4,3151,17849
5,3152,17848


## 3. DTI 上下界（与 04 相同）

In [3]:
DTI_BOUNDS = {
    **{str(value): (float(value), float(value)) for value in range(36, 50)},
    '20%-<30%': (20.0, 30.0),
    '30%-<36%': (30.0, 36.0),
    '50%-60%': (50.0, 60.0),
    '<20%': (np.nan, 20.0),
    '>60%': (60.0, np.nan),
    'MISSING': (np.nan, np.nan),
}


def add_dti_bounds(frame):
    unknown = set(frame['debt_to_income_ratio'].unique()) - set(DTI_BOUNDS)
    require(not unknown, f'DTI 出现未知类别：{sorted(map(str, unknown))}')
    result = frame.copy()
    pairs = frame['debt_to_income_ratio'].map(DTI_BOUNDS)
    result['dti_lower'] = pairs.map(lambda pair: pair[0]).astype('float64')
    result['dti_upper'] = pairs.map(lambda pair: pair[1]).astype('float64')
    require(result.columns.tolist() == FEATURE_COLUMNS + ['dti_lower', 'dti_upper'],
            'DTI 特征列或顺序错误')
    require(result['debt_to_income_ratio'].equals(frame['debt_to_income_ratio']),
            '原始 DTI 类别被修改')
    return result


X_dti = add_dti_bounds(X_base)
X_test_dti = add_dti_bounds(X_test_base)
print(f'候选特征：原始 56 列 + 2 列 DTI 上下界 = {X_dti.shape[1]} 列。')
display(X_dti[['debt_to_income_ratio', 'dti_lower', 'dti_upper']].head())

候选特征：原始 56 列 + 2 列 DTI 上下界 = 58 列。


,debt_to_income_ratio,dti_lower,dti_upper
0,20%-<30%,20.0,30.0
1,49,49.0,49.0
2,20%-<30%,20.0,30.0
3,44,44.0,44.0
4,<20%,NaN,20.0


## 4. 第一轮特征块（与 10 相同）

In [4]:
def safe_ratio(numerator, denominator):
    valid = numerator.notna() & denominator.notna() & (denominator > 0)
    return (numerator / denominator.where(valid)).where(valid).astype('float64')


def build_blocks(train, test):
    both = pd.concat([train, test], ignore_index=True)
    blocks = {}
    blocks['dti'] = add_dti_bounds(both)[['dti_lower', 'dti_upper']]

    ltv_calc = 100 * safe_ratio(both['loan_amount'], both['property_value'])
    blocks['ltv'] = pd.DataFrame({
        'ltv_calc': ltv_calc,
        'ltv_gap': both['loan_to_value_ratio'] - ltv_calc,
    })

    income_dollars = both['income'] * 1000      # HMDA 收入单位为千美元
    blocks['income'] = pd.DataFrame({
        'income_to_area': safe_ratio(income_dollars, both['ffiec_msa_md_median_family_income']),
        'loan_to_income': safe_ratio(both['loan_amount'], income_dollars),
        'value_to_income': safe_ratio(both['property_value'], income_dollars),
    })

    missing = pd.DataFrame({
        'ltv': both['loan_to_value_ratio'].isna(),
        'pv': both['property_value'].isna(),
        'inc': both['income'].isna(),
        'term': both['loan_term'].isna(),
        'dti': both['debt_to_income_ratio'].eq('MISSING'),
        'age': both['applicant_age'].eq('MISSING'),
    })
    blocks['missing'] = pd.DataFrame({
        'missing_pattern': 'm' + missing.astype(int).astype(str).agg(''.join, axis=1),
        'missing_count': missing.sum(axis=1).astype('float64'),
    })

    freq = {}
    for column in ['county_code', 'derived_msa-md', 'state_code']:
        counts = both[column].map(both[column].value_counts()).astype('float64')
        freq[f'freq_{column}'] = counts.where(both[column].ne('MISSING'))
    blocks['freq'] = pd.DataFrame(freq)

    n_train = len(train)
    out = {}
    for name, frame in blocks.items():
        require(len(frame) == len(both), f'{name} 行数不符')
        out[name] = (frame.iloc[:n_train].reset_index(drop=True),
                     frame.iloc[n_train:].reset_index(drop=True))
    return out


BLOCKS = build_blocks(X_base, X_test_base)
NEW_CATEGORICAL = {'missing_pattern'}


def variant_frames(blocks):
    train_parts = [X_base] + [BLOCKS[b][0] for b in blocks]
    test_parts = [X_test_base] + [BLOCKS[b][1] for b in blocks]
    train = pd.concat(train_parts, axis=1)
    test = pd.concat(test_parts, axis=1)
    require(train.columns.is_unique and train.columns.equals(test.columns), '变体列不一致')
    require(train.iloc[:, :len(FEATURE_COLUMNS)].equals(X_base), '原始 56 列被修改')
    cats = CATEGORICAL_COLUMNS + [c for c in train.columns if c in NEW_CATEGORICAL]
    return train, test, cats


for name, (train_block, _) in BLOCKS.items():
    described = train_block.describe(include='all').T
    print(f'\n[{name}] 缺失率：', train_block.isna().mean().round(3).to_dict())
    display(described.iloc[:, :6])


[dti] 缺失率： {'dti_lower': 0.117, 'dti_upper': 0.117}


,count,mean,std,min,25%,50%
dti_lower,92671.0,39.680947,10.654997,20.0,30.0,42.0
dti_upper,92730.0,41.033290,9.699260,20.0,36.0,40.0



[ltv] 缺失率： {'ltv_calc': 0.045, 'ltv_gap': 0.118}


,count,mean,std,min,25%,50%
ltv_calc,100244.0,82.070292,54.826068,0.302115,74.683544,88.235294
ltv_gap,92643.0,5.568211,439.329812,-7717.790000,-0.719136,0.000000



[income] 缺失率： {'income_to_area': 0.054, 'loan_to_income': 0.053, 'value_to_income': 0.088}


,count,mean,std,min,25%,50%
income_to_area,99343.0,1.563771,3.274112,-6.550349,0.746269,1.108830
loan_to_income,99468.0,2.891506,4.565102,0.002606,1.810560,2.692308
value_to_income,95792.0,3.713296,6.426108,0.009722,2.378641,3.288576



[missing] 缺失率： {'missing_pattern': 0.0, 'missing_count': 0.0}


,count,unique,top,freq,mean,std
missing_pattern,105000,42,m000000,88200,NaN,NaN
missing_count,105000.0,NaN,NaN,NaN,0.336019,0.956439



[freq] 缺失率： {'freq_county_code': 0.013, 'freq_derived_msa-md': 0.0, 'freq_state_code': 0.008}


,count,mean,std,min,25%,50%
freq_county_code,103661.0,364.801787,470.338073,1.0,67.0,200.0
freq_derived_msa-md,105000.0,4000.393038,7326.553843,2.0,353.0,927.0
freq_state_code,104176.0,6209.485976,4859.097575,2.0,2667.0,4441.0


## 5. 第二轮特征块（与 12 相同）

In [5]:
def read_raw_ltv(path):
    text = pd.read_csv(path, usecols=['loan_to_value_ratio'], dtype=str,
                       keep_default_na=False)['loan_to_value_ratio']
    return text


raw_ltv = pd.concat([read_raw_ltv(PREPARED_DIR / 'train_features.csv'),
                     read_raw_ltv(PREPARED_DIR / 'test_features.csv')], ignore_index=True)
both = pd.concat([X_base, X_test_base], ignore_index=True)
require(len(raw_ltv) == len(both), 'LTV 文本行数不符')
require(pd.to_numeric(raw_ltv.replace('', np.nan)).equals(both['loan_to_value_ratio'])
        or np.allclose(pd.to_numeric(raw_ltv.replace('', np.nan)), both['loan_to_value_ratio'],
                       equal_nan=True), 'LTV 文本与数值不一致')

decimals = raw_ltv.str.split('.').str[1].fillna('')
precision = np.select(
    [raw_ltv.eq(''), decimals.isin(['', '0']), decimals.str.len().eq(1),
     decimals.str.len().eq(2), decimals.str.len().eq(3)],
    ['missing', 'integer', 'dec1', 'dec2', 'dec3'], default='dec4plus',
)
ltv = both['loan_to_value_ratio']
band = np.select(
    [ltv.isna(), ltv < 80, ltv == 80, ltv < 95, ltv < 97, ltv < 100, ltv == 100],
    ['missing', 'lt80', 'eq80', '80_95', '95_97', '97_100', 'eq100'], default='gt100',
)
PROGRAM_MAX_LTV = {'1': 97.0, '2': 96.5, '3': 100.0, '4': 100.0}   # 常规/FHA/VA/RHS
unknown_types = set(both['loan_type']) - set(PROGRAM_MAX_LTV)
require(not unknown_types, f'未知 loan_type：{sorted(unknown_types)}')
implied_value = safe_ratio(both['loan_amount'] * 100, ltv)
term = both['loan_term']

round2 = {
    'precision': pd.DataFrame({'ltv_precision': precision}),
    'thresholds': pd.DataFrame({
        'ltv_band': band,
        'ltv_over_program': ltv - both['loan_type'].map(PROGRAM_MAX_LTV).astype('float64'),
    }),
    'implied': pd.DataFrame({
        'implied_property_value': implied_value,
        'implied_value_ratio': safe_ratio(implied_value, both['property_value']),
    }),
    'term': pd.DataFrame({
        'term_is_360': term.eq(360).astype('float64').where(term.notna()),
        'term_years': (term / 12).round(1),
    }),
}
n_train = len(X_base)
for name, frame in round2.items():
    require(len(frame) == len(both), f'{name} 行数不符')
    BLOCKS[name] = (frame.iloc[:n_train].reset_index(drop=True),
                    frame.iloc[n_train:].reset_index(drop=True))
NEW_CATEGORICAL |= {'ltv_precision', 'ltv_band'}

train_labels_check = y.to_numpy()
for column, frame in [('ltv_precision', round2['precision']), ('ltv_band', round2['thresholds'])]:
    table = pd.crosstab(frame[column].iloc[:n_train], train_labels_check, normalize='index')
    counts = frame[column].iloc[:n_train].value_counts()
    display(table.assign(rows=counts).round(3))
for name in ['implied', 'term']:
    display(BLOCKS[name][0].describe().T.round(3))
del both

col_0,0,1,rows
ltv_precision,,,
dec1,0.125,0.875,12131
dec2,0.156,0.844,9717
dec3,0.125,0.875,19531
dec4plus,0.154,0.846,306
integer,0.093,0.907,51441
missing,0.462,0.538,11874


col_0,0,1,rows
ltv_band,,,
80_95,0.122,0.878,14329
95_97,0.117,0.883,20177
97_100,0.129,0.871,6015
eq100,0.107,0.893,7295
eq80,0.081,0.919,13455
gt100,0.176,0.824,5711
lt80,0.096,0.904,26144
missing,0.462,0.538,11874


,count,mean,std,min,25%,50%,75%,max
implied_property_value,93126.0,526207.615,2740245.581,0.011,243523.316,367875.648,554999.602,6.270987e+08
implied_value_ratio,92643.0,1.051,2.960,0.003,0.989,1.000,1.009,1.108650e+02


,count,mean,std,min,25%,50%,75%,max
term_is_360,101693.0,0.841,0.366,0.0,1.0,1.0,1.0,1.0
term_years,101693.0,28.146,5.534,0.1,30.0,30.0,30.0,40.0


## 6. 五折训练 + 测试预测（可续跑，与 11 相同）

In [6]:
fixed_config(OUTPUT_DIR, {
    'purpose': 'params x feature blocks, 5-fold with inner early stop; fold-averaged test predictions',
    'seed': SEED, 'inner_stop_fraction': INNER_STOP_FRACTION,
    'max_iterations': MAX_ITERATIONS, 'od_wait': OD_WAIT,
    'gpu_base': {k: v for k, v in GPU_BASE.items() if k != 'thread_count'},
    'data_hashes': DATA_HASHES,
})


def model_params(spec, iterations):
    params = {k: v for k, v in spec.items() if k not in {'early_stop', 'iterations'}}
    return dict(GPU_BASE, **params, iterations=iterations)


def run_one(name, group, fold):
    spec = group['params']
    train, test, cats = variant_frames(group['blocks'])
    fit_idx = np.flatnonzero(fold_assignment != fold)
    valid_idx = np.flatnonzero(fold_assignment == fold)
    started = time.perf_counter()
    best_iteration, early_stopped = None, None
    if spec.get('early_stop', True):
        inner_fit, inner_stop = train_test_split(
            fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED,
            stratify=y.iloc[fit_idx],
        )
        probe = catboost.CatBoostClassifier(**model_params(spec, MAX_ITERATIONS),
                                            eval_metric='Logloss', od_type='Iter',
                                            od_wait=OD_WAIT, use_best_model=True)
        probe.fit(train.iloc[inner_fit], y.iloc[inner_fit], cat_features=cats,
                  eval_set=(train.iloc[inner_stop], y.iloc[inner_stop]))
        best_iteration = int(probe.get_best_iteration()) + 1
        early_stopped = int(probe.tree_count_) < MAX_ITERATIONS
        if not early_stopped:
            print(f'  注意：{name} 到 {MAX_ITERATIONS} 轮仍未早停')
        rounds = int(round(best_iteration / (1 - INNER_STOP_FRACTION)))
        del probe
        gc.collect()
    else:
        rounds = int(spec['iterations'])
    model = catboost.CatBoostClassifier(**model_params(spec, rounds), use_best_model=False)
    model.fit(train.iloc[fit_idx], y.iloc[fit_idx], cat_features=cats)
    require(int(model.tree_count_) == rounds, '外层重训轮数不符')
    valid_prob = checked_probabilities(model, train.iloc[valid_idx])
    test_prob = checked_probabilities(model, test)
    actual = y.iloc[valid_idx]
    stem = OUTPUT_DIR / f'{name}__fold{fold}'
    pd.DataFrame({'id': train_ids.iloc[valid_idx].to_numpy(), 'accepted': actual.to_numpy(),
                  'prob': valid_prob}).to_csv(f'{stem}__valid.csv', index=False, float_format='%.12f')
    pd.DataFrame({'id': test_ids.to_numpy(), 'accepted': test_prob}).to_csv(
        f'{stem}__test.csv', index=False, float_format='%.12f')
    result = {
        'group': name, 'fold': fold, 'definition': group,
        'feature_count': train.shape[1],
        'inner_best_iteration': best_iteration, 'early_stopped': early_stopped,
        'refit_rounds': rounds,
        'valid_auc': roc_auc_score(actual, valid_prob),
        'valid_logloss': log_loss(actual, valid_prob, labels=[0, 1]),
        'seconds': time.perf_counter() - started, 'gpu_name': GPU_NAME,
        'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    write_json(Path(f'{stem}.json'), result)   # 最后写 json，作为该折完成的标志
    del model, train, test
    gc.collect()
    return result


selected = {n: g for n, g in GROUPS.items() if RUN_ONLY is None or n in RUN_ONLY}
for fold in FOLDS:
    for name, group in selected.items():
        result_path = OUTPUT_DIR / f'{name}__fold{fold}.json'
        if result_path.is_file():
            saved = json.loads(result_path.read_text(encoding='utf-8'))
            require(saved['definition'] == group, f'{name} 的定义已改变；请改组名')
            print(f'{name} fold {fold}: 已完成 AUC={saved["valid_auc"]:.6f}，跳过')
            continue
        result = run_one(name, group, fold)
        print(f'{name} fold {fold}: AUC={result["valid_auc"]:.6f}  '
              f'轮数={result["refit_rounds"]}  {result["seconds"]/60:.1f} 分钟', flush=True)

M0_C11_all fold 1: AUC=0.912896  轮数=4138  7.1 分钟
M1_C11_G7 fold 1: AUC=0.914112  轮数=4053  6.6 分钟
M2_C7_G7 fold 1: AUC=0.914422  轮数=2945  7.0 分钟
M3_C11_lean fold 1: AUC=0.914450  轮数=4069  6.6 分钟
M0_C11_all fold 2: AUC=0.927412  轮数=5802  9.2 分钟
M1_C11_G7 fold 2: AUC=0.926594  轮数=5662  9.2 分钟
M2_C7_G7 fold 2: AUC=0.926109  轮数=3787  9.0 分钟
M3_C11_lean fold 2: AUC=0.927037  轮数=5689  9.2 分钟
M0_C11_all fold 3: AUC=0.920058  轮数=5174  8.2 分钟
M1_C11_G7 fold 3: AUC=0.921108  轮数=4384  7.1 分钟
M2_C7_G7 fold 3: AUC=0.921391  轮数=3011  7.2 分钟
M3_C11_lean fold 3: AUC=0.920750  轮数=3289  5.3 分钟
M0_C11_all fold 4: AUC=0.915731  轮数=3934  6.2 分钟
M1_C11_G7 fold 4: AUC=0.916444  轮数=6202  10.1 分钟
M2_C7_G7 fold 4: AUC=0.917070  轮数=2359  5.7 分钟
M3_C11_lean fold 4: AUC=0.916733  轮数=4981  8.0 分钟
M0_C11_all fold 5: AUC=0.920443  轮数=5652  9.0 分钟
M1_C11_G7 fold 5: AUC=0.921181  轮数=5195  8.3 分钟
M2_C7_G7 fold 5: AUC=0.920948  轮数=2836  6.7 分钟
M3_C11_lean fold 5: AUC=0.921801  轮数=5195  8.3 分钟


## 7. 五折比较、融合与提交文件

每个完成五折的组写出 `submission_<组>.csv`；融合组写出 `submission_blend_M1_M2_M3.csv`。

In [7]:
results = pd.DataFrame([json.loads(p.read_text(encoding='utf-8'))
                        for p in sorted(OUTPUT_DIR.glob('*__fold?.json'))])
require(not results.empty, '还没有完成的结果')
auc = results.pivot(index='fold', columns='group', values='valid_auc')
require(BASELINE in auc.columns, f'缺少对照 {BASELINE}')


def load_complete(name):
    folds = sorted(results.loc[results['group'].eq(name), 'fold'])
    if folds != [1, 2, 3, 4, 5]:
        return None
    valid = pd.concat([pd.read_csv(OUTPUT_DIR / f'{name}__fold{k}__valid.csv').assign(fold=k)
                       for k in folds]).sort_values('id', kind='stable').reset_index(drop=True)
    tests = [pd.read_csv(OUTPUT_DIR / f'{name}__fold{k}__test.csv', dtype={'id': str}) for k in folds]
    for frame in tests:
        require(frame['id'].equals(sample_submission['id']), f'{name} 测试 ID 顺序不一致')
    return valid, np.mean([t['accepted'].to_numpy() for t in tests], axis=0)


def write_submission(name, probability):
    require(np.isfinite(probability).all() and len(probability) == 45000, '提交概率无效')
    path = OUTPUT_DIR / f'submission_{name}.csv'
    pd.DataFrame({'id': sample_submission['id'], 'accepted': probability}).to_csv(
        path, index=False, float_format='%.12f')
    return path.name


complete = {name: load_complete(name) for name in auc.columns}
complete = {k: v for k, v in complete.items() if v is not None}
rows = []
for name in auc.columns:
    row = {'group': name, 'folds_done': int(auc[name].notna().sum()),
           'mean_auc': auc[name].mean(),
           'mean_delta_vs_M0': (auc[name] - auc[BASELINE]).mean(),
           'folds_better_than_M0': int(((auc[name] - auc[BASELINE]) > 0).sum()),
           'mean_refit_rounds': results.loc[results['group'].eq(name), 'refit_rounds'].mean()}
    if name in complete:
        valid, test_prob = complete[name]
        row['oof_auc'] = roc_auc_score(valid['accepted'], valid['prob'])
        row['submission'] = write_submission(name, test_prob)
    rows.append(row)

if all(name in complete for name in BLEND_GROUPS):
    base_valid = complete[BLEND_GROUPS[0]][0]
    for name in BLEND_GROUPS[1:]:
        require(complete[name][0]['id'].equals(base_valid['id']), '融合 OOF 的 ID 不一致')
    blend_prob = np.mean([complete[n][0]['prob'].to_numpy() for n in BLEND_GROUPS], axis=0)
    fold_auc = pd.Series({k: roc_auc_score(base_valid.loc[base_valid['fold'].eq(k), 'accepted'],
                                           blend_prob[base_valid['fold'].eq(k).to_numpy()])
                          for k in range(1, 6)})
    delta = fold_auc - auc[BASELINE]
    rows.append({'group': 'BLEND_' + '+'.join(BLEND_GROUPS), 'folds_done': 5,
                 'mean_auc': fold_auc.mean(), 'mean_delta_vs_M0': delta.mean(),
                 'folds_better_than_M0': int((delta > 0).sum()),
                 'oof_auc': roc_auc_score(base_valid['accepted'], blend_prob),
                 'submission': write_submission(
                     'blend_M1_M2_M3',
                     np.mean([complete[n][1] for n in BLEND_GROUPS], axis=0))})

table = pd.DataFrame(rows).set_index('group').sort_values('mean_auc', ascending=False)
table.to_csv(OUTPUT_DIR / 'summary.csv')
display(table.round(6))
display(auc.sub(auc[BASELINE], axis=0).round(6))
done = table[table['folds_done'] == 5]
if not done.empty:
    best = done['mean_auc'].idxmax()
    print(f'五折平均 AUC 最高：{best}（{done.loc[best, "mean_auc"]:.6f}）→ 提交 {done.loc[best, "submission"]}')

,folds_done,mean_auc,mean_delta_vs_M0,folds_better_than_M0,mean_refit_rounds,oof_auc,submission
group,,,,,,,
BLEND_M1_C11_G7+M2_C7_G7+M3_C11_lean,5,0.920508,0.001200,4,NaN,0.920514,submission_blend_M1_M2_M3.csv
M3_C11_lean,5,0.920154,0.000846,4,4644.6,0.920132,submission_M3_C11_lean.csv
M2_C7_G7,5,0.919988,0.000680,4,2987.6,0.920013,submission_M2_C7_G7.csv
M1_C11_G7,5,0.919888,0.000580,4,5099.2,0.919853,submission_M1_C11_G7.csv
M0_C11_all,5,0.919308,0.000000,0,4940.0,0.919344,submission_M0_C11_all.csv


group,M0_C11_all,M1_C11_G7,M2_C7_G7,M3_C11_lean
fold,,,,
1,0.0,0.001215,0.001526,0.001554
2,0.0,-0.000818,-0.001303,-0.000375
3,0.0,0.001050,0.001332,0.000692
4,0.0,0.000714,0.001339,0.001003
5,0.0,0.000739,0.000505,0.001358


五折平均 AUC 最高：BLEND_M1_C11_G7+M2_C7_G7+M3_C11_lean（0.920508）→ 提交 submission_blend_M1_M2_M3.csv
